# Install Required Software or Run a Docker Image

<br>
    
This notebook installs the Pixi software environment required to run the other notebooks in the NISAR GCOV Cookbook.

<hr>

## Overview

1. [Prerequisites](env-prereqs)
1. [(Option 1) Install the environment with the Pixi pacakage manager](env-pixi)
1. [(Option 2) Run a Docker Image with the software environment already installed](env-docker)
1. [Summary](env-summary)
1. [Resources and references](env-resources)

<hr>

(env-prereqs)=
## 1. Prerequisites

| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [Option 1: The Pixi package manager must be installed on the system running Jupyter Lab](https://pixi.sh/latest/installation/) | Necessary | Pixi may not be supported on all Jupyter Hubs|
| [Option 2: Docker must be installed and running on the system running Jupyter Lab](https://www.docker.com/get-started/) | Necessary | Run the Docker image locally or in a Jupyter Hub |
| [UW Scientific Software Engineering Center's Pixi guide](https://rse-guidelines.readthedocs.io/en/latest/fundamentals/computing-development-environments/pixi/) | Helpful | This is a great resource to get quickly get started with Pixi|

- **Rough Notebook Time Estimate**: 3 minutes

<hr>

(env-pixi)=
## 2. (Option 1) Install the environment with the Pixi pacakage manager

### 2a. Create a `work_dir` context manager

This is useful when you want to run a command in a working directory and then automatically change back to your original directory 

In [ ]:
import os
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def work_dir(new_dir):
    old_dir = os.getcwd()
    os.chdir(new_dir)
    try:
        yield
    finally:
        os.chdir(old_dir)

### 2b. Install the `isce3` environment with Pixi

In [ ]:
with work_dir(Path.cwd().parent):
    !pixi install -e isce3

### 2c. Register the `isce3` environment's Python kernel with `ipykernel`

In [ ]:
env_name = "isce3"
display_name = f'"{env_name} (Python)"'

!pixi run -e isce3 python -m ipykernel install \
  --user \
  --name $env_name \
  --display-name $display_name

### 2d. Ensure notebook shell commands run in the `isce3` environment

This ensures that shell commands executed from inside a notebook with `!` run in the notebook kernel’s environment. This works by launching the entire Jupyter kernel process inside the Pixi environment, so the kernel’s PATH, environment variables, and Python executable all come from that Pixi environment, not from the parent JupyterLab environment.

In [ ]:
from jupyter_client.kernelspec import KernelSpecManager
import json

ksm = KernelSpecManager()
spec = ksm.get_kernel_spec(env_name)
kernel_dir = Path(spec.resource_dir)
kernel_json = kernel_dir / "kernel.json"

data = json.loads(kernel_json.read_text())
orig_argv = data.get("argv", [])

new_argv = [
    "pixi",
    "run",
    "--manifest-path",
    str(Path.cwd().parent),
    "-e",
    env_name,
] + orig_argv

data["argv"] = new_argv
kernel_json.write_text(json.dumps(data, indent=2))
print(f"Updated kernel.json at {kernel_json} with Pixi wrapper.")
print("argv:", data["argv"])

### 2e. (Optional) Delete the environment and remove its `kernelspec`

In [ ]:
# Uncomment and run the code below to delete your pixi environment

# with work_dir(Path.cwd().parent):
#     !pixi clean
#     !jupyter kernelspec remove isce3 -y

<hr>

(env-docker)=
## 3. (Option 2) Run a Docker Image with the software environment already installed

### 3a. Verify Docker is installed and working

In a terminal (PowerShell on Windows, or a regular terminal on macOS/Linux), confirm Docker is installed and the daemon is running:

```
docker --version
docker run hello-world
```

If `hello-world` runs successfully and prints a confirmation message, Docker is ready to use.

### 3b. Create a Dockerfile

Create a new, empty folder for this build (for example, `isce3-docker`), and `cd` into it. Inside that folder, create a file named exactly `Dockerfile` (no file extension) containing the following:

```dockerfile
FROM condaforge/miniforge3:latest

RUN conda create -n isce3 -c conda-forge python=3.11 isce3-cpu jupyterlab -y

WORKDIR /workspace

EXPOSE 8888

ENTRYPOINT ["conda", "run", "--no-capture-output", "-n", "isce3", "jupyter", "lab", \
    "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", \
    "--ServerApp.token=", "--ServerApp.password="]
```

**On Windows**, if you're creating this file from PowerShell rather than a text editor, use the following to avoid encoding issues (e.g. Notepad's default save encoding can produce a file Docker can't read):

```powershell
@'
FROM condaforge/miniforge3:latest

RUN conda create -n isce3 -c conda-forge python=3.11 isce3-cpu jupyterlab -y

WORKDIR /workspace

EXPOSE 8888

ENTRYPOINT ["conda", "run", "--no-capture-output", "-n", "isce3", "jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", "--ServerApp.token=", "--ServerApp.password="]
'@ | Set-Content -Encoding ascii -NoNewline Dockerfile
```

If using a text editor instead, make sure "Save as type" is set to **All Files** (not `.txt`) so the filename stays exactly `Dockerfile`.

### 3c. Build the image

From inside the folder containing your `Dockerfile`, run:

```
docker build -t isce3-env .
```

This downloads the Miniforge base image and installs `isce3-cpu` and JupyterLab into a conda environment named `isce3` inside the image.

### 3d. Run the container

Run the image, giving it a distinctive name and mounting the folder containing your cloned cookbook repository (e.g. `NISAR_Cookbook`) into the container's `/workspace` directory so your notebooks and any outputs persist on your local machine. Binding to `127.0.0.1` (rather than all interfaces) keeps the unauthenticated server local-only:

**Windows (PowerShell):**
```powershell
docker run --name isce3-jupyter -p 127.0.0.1:8888:8888 -v C:\path\to\NISAR_Cookbook:/workspace isce3-env
```

**macOS/Linux:**
```bash
docker run --name isce3-jupyter -p 127.0.0.1:8888:8888 -v /path/to/NISAR_Cookbook:/workspace isce3-env
```

Replace the path before the colon with the actual location of your cloned `NISAR_Cookbook` repository. This starts JupyterLab and prints a URL of the form:

```
http://127.0.0.1:8888/lab
```

Open that URL in your browser.

### 3e. Verify the environment

Run the following cell to confirm you're in the containerized Linux environment and that `isce3` imports correctly:

In [ ]:
import platform
print(platform.system())

import isce3
print(isce3.__version__)

### Troubleshooting Tips
#### Stop and Start Docker
To stop the container, press `Ctrl+C` in the terminal running it, or run `docker stop isce3-jupyter` from a new terminal.
To resume the **same** container later (keeping any packages or files inside it, outside of `/workspace`, **do not** run `docker run` again (if you do, it creates a separate new container.) Use `docker start -a isce3-jupyter`

#### Docker Setup - "virtualization not detected"
In Windows, you may encounter Docker Desktop reporting that virtualization support isn't detected. If so, you can run the following in PowerShell **as an Administrator**, then restart your computer. 
`dism.exe /online /enable-feature /featurename:Microsoft-Windows-Subsystem-Linux /all /norestart`
`dism.exe /online /enable-feature /featurename:VirtualMachinePlatform /all /norestart`

You can confirm virtualization is enabled by going to Task Manager >> Performance >> CPU, where "Virtualization" should be Enabled.

#### Jupyter Requires Authentication
Some environments enforce policies that require authentication. If a login page appears depsite setting `--ServerApp.token=`/`--ServerApp.password=` in 3b. 

1. Get the login token from the container's startup logs: `docker logs isce3-jupyter`. Copy the full URL into the browser
2. Try an incognite browser
3. Set a password `docker exec -it isce3-jupyter conda run -n isce3 jupyter server password`. Stop and restart the container, then reload http://127.0.0.1:8888/lab and log in with that password

(env-summary)=
## 4. Summary
Now that you have installed the software environment or have the Docker image running, [make sure you have access to the data](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/set_up_Earthdata_Login.md). You will then be ready to run the remaining notebooks in the NISAR GCOV Cookbook.
<hr>

(env-resources)=
## 5. Resources and references

### References
- [{abbr}`UW SSEC (Univertsity of Washington Scientific Software Engineering Center)`](https://escience.washington.edu/software-engineering/ssec/)

**Author:** 
Alex Lewandowski,
Tracy Tien